# Limpieza y ingeniería de atributos
Este notebook va a ser para limpiar el dataframe.

Se recibe como entrada un dataframe (puede ser el de cualquier año o el consolidado total) y se van a procesar y limpiar las siguientes columnas:

- Género: Poner únicamente los géneros en F, M, No especificado
- Edad: Verificar que no haya edades negativas o que no cuadren (muy bajas o muy altas)
- Fechas: Verificar que estén en formato DD/MM/AAAA
- Eliminar nulos y duplicados

# Bibliotecas

In [2]:
%pip install polars

   ---------------------------------------- 0.0/783.6 kB ? eta -:--:--
   ---------------------------------------- 783.6/783.6 kB 8.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   -- ------------------------------------- 2.4/41.3 MB 11.7 MB/s eta 0:00:04
   ---- ----------------------------------- 4.7/41.3 MB 11.3 MB/s eta 0:00:04
   ------ --------------------------------- 6.6/41.3 MB 10.9 MB/s eta 0:00:04
   -------- ------------------------------- 8.7/41.3 MB 10.6 MB/s eta 0:00:04
   ---------- ----------------------------- 10.7/41.3 MB 10.5 MB/s eta 0:00:03
   ------------ --------------------------- 12.8/41.3 MB 10.3 MB/s eta 0:00:03
   -------------- ------------------------- 14.9/41.3 MB 10.3 MB/s eta 0:00:03
   ---------------- ----------------------- 17.0/41.3 MB 10.3 MB/s eta 0:00:03
   ------------------ --------------------- 18.6/41.3 MB 10.1 MB/s eta 0:00:03
   ------------------- -------------------- 20.2/41.3 MB 9.8 MB/s eta 0:


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import polars as pl
import glob

# Obtenemos los datos de entrada

In [3]:
# 1. Encontrar todos los paths de tus archivos
file_paths = glob.glob("DATA_Actual/*.parquet")

lazy_frames = [] # Una lista para guardar las "recetas"

# 2. Iterar sobre cada path
for path in file_paths:
    # Escanear UN solo archivo (sigue siendo lazy)
    lf = pl.scan_parquet(path)
    
    # AÑADIR LA INSTRUCCIÓN: Forzar la columna problemática a Texto (Utf8)
    # Esto unifica el schema de todos los archivos ANTES de unirlos.
    lf_corrected = lf.with_columns(
        pl.col("Edad_Usuario").cast(pl.Utf8),
        pl.col("Bici").cast(pl.Utf8)
        # ...si otra columna da error de schema, la añadimos aquí, no metemos todo de golpe porque se muere el kernel U_U
    )
    
    # Guardar la "receta" corregida en nuestra lista
    lazy_frames.append(lf_corrected)

# 3. Concatenar verticalmente TODAS las "recetas" en una sola
df_lazy = pl.concat(lazy_frames, how="vertical")

print(f"¡Éxito! {len(lazy_frames)} archivos han sido escaneados y unidos en un solo LazyFrame.")

¡Éxito! 2 archivos han sido escaneados y unidos en un solo LazyFrame.


# Eliminamos nulos

In [4]:
# Esta celda rellena "No especificado" en TODAS las columnas de texto
# (incluyendo Edad_Usuario por ahora, pero no importa)
df_lazy = df_lazy.with_columns(
    pl.col(pl.Utf8).fill_null("No especificado")
)

# Eliminamos duplicados

In [5]:
df_lazy = df_lazy.unique()

# Pasamos a entero la columna de edades

In [6]:
# Como todo es texto, lo pasamos a Float (para "16.0") y luego a Int
# El "strict=False" convertirá "No especificado" a null
df_lazy = df_lazy.with_columns(
    pl.col("Edad_Usuario").cast(pl.Float64, strict=False)
)
df_lazy = df_lazy.with_columns(
    pl.col("Edad_Usuario").cast(pl.Int32, strict=False)
)

In [7]:
# Verificamos que ya no haya nulos
# Esta celda AHORA SÍ va a funcionar
print("Calculando conteo de nulos...")

null_counts = df_lazy.select(
    pl.all().null_count()
).collect()

print(null_counts)

Calculando conteo de nulos...
shape: (1, 10)
┌────────────┬────────────┬──────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ Genero_Usu ┆ Edad_Usuar ┆ Bici ┆ Ciclo_Esta ┆ … ┆ Ciclo_Esta ┆ Fecha_Arr ┆ Hora_Arri ┆ _source_f │
│ ario       ┆ io         ┆ ---  ┆ cion_Retir ┆   ┆ cion_Arrib ┆ ibo       ┆ bo        ┆ ile       │
│ ---        ┆ ---        ┆ u32  ┆ o          ┆   ┆ o          ┆ ---       ┆ ---       ┆ ---       │
│ u32        ┆ u32        ┆      ┆ ---        ┆   ┆ ---        ┆ u32       ┆ u32       ┆ u32       │
│            ┆            ┆      ┆ u32        ┆   ┆ u32        ┆           ┆           ┆           │
╞════════════╪════════════╪══════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 0          ┆ 1586       ┆ 0    ┆ 0          ┆ … ┆ 0          ┆ 0         ┆ 0         ┆ 0         │
└────────────┴────────────┴──────┴────────────┴───┴────────────┴───────────┴───────────┴───────────┘


# Tratamiento de columna de edades

En esta sección vamos a limpiar las edades. Primero vamos a hacer un describe() para ver si los límites inferior y superior son muy atípicos, así como comparar el sesgo existente entre la media y la mediana (si existe). Después, a los valores atipicos, los reemplazaremos por la mediana, para no considerar sesgos, si los hay, y tener todos nuestros registros con valores normalizados

In [8]:
# Vemos la columna de Edad, para saber los umbrales máximos y mínimos a establecer
descripcion_edad = df_lazy.select(
    "Edad_Usuario"
).describe()

print("Estadísticas de la columna 'Edad_Usuario':")
print(descripcion_edad)

Estadísticas de la columna 'Edad_Usuario':
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ Edad_Usuario │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 3.7576021e7  │
│ null_count ┆ 1586.0       │
│ mean       ┆ 33.842496    │
│ std        ┆ 9.724336     │
│ min        ┆ 16.0         │
│ 25%        ┆ 27.0         │
│ 50%        ┆ 32.0         │
│ 75%        ┆ 39.0         │
│ max        ┆ 160.0        │
└────────────┴──────────────┘


In [9]:
# 1. Definir el rango válido de edades
EDAD_MIN = 16
EDAD_MAX = 80

# 2. Calcular la mediana SÓLO de las edades válidas
#    .compute() ejecuta solo este pequeño cálculo para obtener el valor
mediana_edad_valida = df_lazy.select(
    pl.col("Edad_Usuario")
      .filter(pl.col("Edad_Usuario").is_between(EDAD_MIN, EDAD_MAX))
      .median()
).collect().item() # .item() saca el valor (ej: 34.0)

# Convertimos la mediana a entero por si acaso
mediana_edad_valida = int(mediana_edad_valida)

print(f"La mediana de edad válida es: {mediana_edad_valida}")

# 3. Aplicar la limpieza al LazyFrame:
df_lazy = df_lazy.with_columns(
    pl.when(pl.col("Edad_Usuario").is_between(EDAD_MIN, EDAD_MAX))
      .then(pl.col("Edad_Usuario")) # Si la edad es válida, déjala como está
      .otherwise(mediana_edad_valida)   # Si es nula o está fuera de rango, pon la mediana
      .alias("Edad_Usuario")          # Sobrescribe la columna original
)

print("Limpieza de 'Edad_Usuario' (rango 16-80 e imputación de nulos) aplicada al LazyFrame.")

La mediana de edad válida es: 32
Limpieza de 'Edad_Usuario' (rango 16-80 e imputación de nulos) aplicada al LazyFrame.


# Tratamiento a columna "Genero_Usuario"
Solo vamos a tomar en cuenta los registros que sean 'F', 'M' y 'No especificado'. Los registros que no sean 'F' o 'M', serán reemplazados por 'No especificado'.

In [10]:
df_lazy = df_lazy.with_columns(
    pl.when(pl.col("Genero_Usuario") == 'M').then(pl.lit('M'))
      .when(pl.col("Genero_Usuario") == 'F').then(pl.lit('F'))
      .otherwise(pl.lit('No especificado'))
      .alias("Genero_Usuario")
)

print("Regla de limpieza para 'Genero_Usuario' aplicada al LazyFrame.")

Regla de limpieza para 'Genero_Usuario' aplicada al LazyFrame.


In [11]:
print("\nValores de 'Genero_Usuario' después de la limpieza:")

# Usamos .value_counts("nombre_columna") y luego .collect()
counts = df_lazy.group_by("Genero_Usuario").agg(pl.count().alias("counts")).collect()

print(counts)


Valores de 'Genero_Usuario' después de la limpieza:


C:\Users\miste\AppData\Local\Temp\ipykernel_13604\3984967311.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  counts = df_lazy.group_by("Genero_Usuario").agg(pl.count().alias("counts")).collect()


shape: (3, 2)
┌─────────────────┬──────────┐
│ Genero_Usuario  ┆ counts   │
│ ---             ┆ ---      │
│ str             ┆ u32      │
╞═════════════════╪══════════╡
│ M               ┆ 25851267 │
│ F               ┆ 10669716 │
│ No especificado ┆ 1056624  │
└─────────────────┴──────────┘


# Tratamiento a columnas de fechas

Algunas fechas vienen con guión en lugar de diagonal, vamos a poner todo en diagonales.

In [12]:
# Reemplaza '-' por '/' en las columnas de fecha
df_lazy = df_lazy.with_columns(
    
    pl.col("Fecha_Retiro").str.replace_all("-", "/"),
    pl.col("Fecha_Arribo").str.replace_all("-", "/")

)

# Exportamos el dataframe limpio para poder trabajarlo en el siguiente notebook: Ingeniería de Atributos

In [ ]:
# Nombre del archivo de salida
output_file = "ecobici_limpio_2024_2025.parquet"

print(f"Iniciando el guardado en '{output_file}'...")
print("Esto puede tardar un poco, Polars está ejecutando toda la limpieza...")

# .sink_parquet() ejecuta el plan lazy y lo guarda
# directamente en el archivo, sin llenar la RAM.
df_lazy.sink_parquet(output_file)

print(f"¡Éxito! Tu DataFrame limpio fue guardado en '{output_file}'")

Iniciando el guardado en 'ecobici_limpio_2023_2024_2025.parquet'...
Esto puede tardar un poco, Polars está ejecutando toda la limpieza...
¡Éxito! Tu DataFrame limpio fue guardado en 'ecobici_limpio_2023_2024_2025.parquet'
